# 03 大模型应用开发 · 第02课：提示词与模型参数

上一课完成了最小模型调用链路。本课不增加复杂API功能，而是研究两个直接影响回答的输入：**提示词怎么写**，以及**模型参数怎么选**。

本课主线：把一次随手提问，改造成结构清楚、可以复用、可以离线测试的请求配置。

## 1. 学完后你能独立完成什么

完成本课后，你应该能够：

1. 区分任务、上下文、约束和输出要求；
2. 用`system`和`user`消息组织一次单轮请求；
3. 解释消息角色由谁定义、内容由谁提供；
4. 解释`model`、推理强度、`temperature`和`max_tokens`分别控制什么；
5. 设计一次只改变一个因素的模型参数对照实验；
6. 用固定测试样例比较提示词版本，而不是凭一次回答下结论；
7. 在不联网的情况下构造并测试请求数据。

本课不讲JSON结构化输出、多轮历史、流式响应和Agent，这些内容留给后续课程。

## 2. 运行本课前：安装必要依赖

在仓库根目录的终端中执行：

```powershell
python -m pip install -r 03_大模型应用开发/02_提示词与模型参数/requirements.txt
```

其中`python`选择解释器，`-m pip`让该解释器运行pip，`install`表示安装，`-r`表示从文件读取依赖。`openai`用于可选的真实请求，`python-dotenv`用于读取本地配置，`pytest`用于课末作业测试。

先运行`Path.cwd()`可以看到Notebook内核的当前工作目录。`Path`来自Python标准库`pathlib`模块，`Path.cwd()`不接收参数，返回表示当前目录的`Path`对象。

如果当前目录就是`02_提示词与模型参数`，可以运行：

```python
%pip install -r requirements.txt
```

如果当前目录是仓库根目录，则运行：

```python
%pip install -r 03_大模型应用开发/02_提示词与模型参数/requirements.txt
```

`%pip`会把依赖安装到当前Notebook内核使用的环境；安装后如果仍提示无法导入，请重启内核。

In [ ]:
import sys
from pathlib import Path

import dotenv
import openai
import pytest

print("当前工作目录：", Path.cwd())
print("Python：", sys.version.split()[0])
print("openai：", openai.__version__)
print("pytest：", pytest.__version__)
print("✅ 本课依赖可以导入。")

## 3. 从“请求能发送”到“请求表达清楚”

上一课的最小请求是：

```python
messages = [{"role": "user", "content": question}]
```

它能够工作，但如果任务复杂，只写一句问题可能留下很多没有说明的选择：回答给谁看、允许依据什么资料、长度是多少、不知道时怎么办。模型只能根据当前输入和训练形成的模式生成结果，不能自动知道你的业务规则。

```text
业务目标
  ↓ 变成明确的提示词和消息
模型参数
  ↓ 与消息一起传给API
模型生成候选回答
  ↓ 用固定样例检查
决定保留或修改提示词版本
```

## 4. 提示词的四个组成部分

**提示词（prompt）**是应用交给模型的指令和信息总和，不只是用户最后输入的那一句话。一个便于检查的提示词通常可以拆成：

| 部分 | 回答的问题 | 示例 |
|---|---|---|
| 任务 | 要模型做什么 | 解释错误信息 |
| 上下文 | 完成任务需要哪些已知信息 | 用户是Python初学者 |
| 约束 | 哪些行为允许或禁止 | 不编造不存在的命令 |
| 输出要求 | 结果应怎样呈现 | 先说原因，再给修改代码 |

这些名称不是Python关键字，也不是API固定字段，而是我们分析提示词时使用的设计框架。不是每个请求都必须写四段，但重要规则应该明确表达。

In [ ]:
task = "解释下面的Python错误"
context = "读者刚学会函数和异常处理"
constraints = "不要假设读者了解SDK内部实现；不确定时明确说明"
output_requirement = "先用一句话说明原因，再给出修改后的代码"
error_text = "AttributeError: 'NoneType' object has no attribute 'strip'"

user_prompt = f"""任务：{task}
上下文：{context}
约束：{constraints}
输出要求：{output_requirement}
待分析内容：{error_text}"""

print(user_prompt)

上面的代码只是在本地构造字符串，没有调用API。`f"""..."""`是多行f-string，花括号中的变量值会被放入文本。

变化情况：简单任务可能只需要任务和输出要求。边界情况：约束互相冲突时，例如同时要求“只回答一句”和“逐行详细解释”，模型无法同时完全满足；这不是增加更多文字就能解决的，需要开发者消除冲突。

提示词能提高任务表达的清晰度，但不能保证事实永远正确，也不能替代权限控制、数据校验和自动测试。

### 课堂练习1：补齐提示词组成部分

**目标**：区分任务、上下文、约束和输出要求。

**题目**：填写下面四个变量，使它们分别承担对应作用。变量名必须保持不变。

**限制**：只构造字符串，不访问网络。

**预期结果**：检测能够在四个变量中找到指定关键信息。


**自检方式**：先运行答题单元，再运行紧随其后的自动检测单元。

In [6]:
exercise_task = "总结会议记录"          # 要求模型总结会议记录
exercise_context = "读者没有技术背景"       # 说明读者没有技术背景
exercise_constraint = "禁止添加记录中不存在的决定"    # 禁止添加记录中不存在的决定
exercise_output = "要求使用三个要点"        # 要求使用三个要点

In [7]:
try:
    _check_values = [exercise_task, exercise_context, exercise_constraint, exercise_output]
    if not all(isinstance(_check_value, str) and _check_value.strip() for _check_value in _check_values):
        print("❌ 四个变量都必须填写为非空字符串。")
    elif "总结" not in exercise_task or "会议" not in exercise_task:
        print("❌ exercise_task需要说明总结会议记录这个任务。")
    elif "技术" not in exercise_context:
        print("❌ exercise_context需要说明读者没有技术背景。")
    elif ("决定" not in exercise_constraint or not any(_check_word in exercise_constraint for _check_word in ("禁止", "不能", "不要"))):
        print("❌ exercise_constraint需要明确禁止添加不存在的决定。")
    elif "三" not in exercise_output and "3" not in exercise_output:
        print("❌ exercise_output需要明确三个要点。")
    else:
        print("✅ 四个提示词组成部分填写正确。")
except NameError:
    print("❌ 请先运行练习1的答题单元。")

✅ 四个提示词组成部分填写正确。


<details><summary>完成练习1后查看参考答案</summary>

```python
exercise_task = "总结下面的会议记录"
exercise_context = "读者没有技术背景"
exercise_constraint = "不要添加会议记录中不存在的决定"
exercise_output = "使用三个要点输出"
```

判断重点不是句子必须逐字相同，而是每段只承担一个清楚的职责。

</details>

## 5. 消息角色：谁在说话，规则放在哪里

Chat Completions API中的每条消息是一个字典。本课使用两个API字段：

| 字段 | 是否是Python关键字 | 由谁定义 | 类型 | 含义 |
|---|---|---|---|---|
| `role` | 否 | API协议 | 字符串 | 消息的来源或作用 |
| `content` | 否 | API协议 | 字符串 | 消息的实际文本 |

字段名称由API协议定义，开发者负责提供本次请求的具体值。

- `system`：放置服务商或应用支持的总体规则、身份和边界；
- `developer`：部分API用它表达由应用开发者提供的规则；它是否可用、与其他角色如何处理冲突，由具体API协议定义；
- `user`：放置本次用户任务和输入；
- `assistant`：表示模型之前的回答，主要在多轮对话中使用，第四课再深入。

`system`、`developer`、`user`可以看作不同来源的指令层，但不能脱离具体API讨论固定优先级。本课实际连接DeepSeek兼容API，因此可运行示例只依赖其常见的`system`和`user`；使用其他服务时必须核对该服务当前支持的角色，不能把某一家服务的角色名称想当然地用于所有API。

In [ ]:
system_prompt = (
    "你是Python学习助手。只解释用户提供的代码；"
    "不确定时明确说明。回答控制在五句话以内。"
)
user_request = "请解释：为什么None不能调用strip()？"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_request},
]

print(messages)

执行时，Python先创建字符串和列表；SDK随后把列表序列化成JSON；远端API验证字段并把消息交给模型。用户代码中没有手写JSON转换行，因为SDK在`create()`内部完成了转换。

正常情况：两个角色职责清楚。变化情况：没有通用规则的简单请求可以只有`user`消息。错误情况：把`role`拼成`roles`或填入服务商不支持的值，API可能拒绝请求。

`system`通常比`user`更适合存放应用规则，但它不是安全沙箱。权限检查、敏感操作确认和输出校验仍必须由应用代码完成。

### 课堂练习2：构造两条消息

**目标**：把应用规则和本次任务放入不同角色。

**题目**：完成`make_messages(rule, request)`。两个参数都由调用者提供：

| 参数 | 类型 | 包含什么内容 | 放到哪里 |
|---|---|---|---|
| `rule` | `str` | 应用对模型提出的规则，例如`"只回答Python问题"` | 第一条消息的`content`，对应`system`角色 |
| `request` | `str` | 用户本次提出的问题，例如`"什么是列表？"` | 第二条消息的`content`，对应`user`角色 |

函数必须先去除两个字符串的首尾空白，再返回下面这种结构的新列表：

```python
[
    {"role": "system", "content": 清理后的rule},
    {"role": "user", "content": 清理后的request},
]
```

如果任意参数不是字符串，或者去除首尾空白后为空，统一抛出`ValueError("消息不能为空")`。

**限制**：不访问网络，不增加第三条消息。

**预期结果**：正常输入被清理，空规则和空请求都会被拒绝。

**自检方式**：先运行答题单元，再运行紧随其后的自动检测单元。

In [14]:
def make_messages(rule: str, request: str) -> list[dict[str, str]]:
    # TODO：检查rule和request，清理文本，再按题目要求返回两条消息
    if not isinstance(rule, str) or not rule.strip():
        raise ValueError("消息不能为空")
    if not isinstance(request, str) or not request.strip():
        raise ValueError("消息不能为空")

    ans = []
    ans.append({
        'role': 'system',
        'content': rule.strip()
    })
    ans.append({
        'role': 'user',
        'content': request.strip()
    })
    return ans


In [15]:
try:
    _check_messages = make_messages("  只回答Python问题  ", "  什么是列表？  ")
    _check_expected = [
        {"role": "system", "content": "只回答Python问题"},
        {"role": "user", "content": "什么是列表？"},
    ]
    if _check_messages != _check_expected:
        print("❌ 应按system、user顺序返回两条清理后的消息。")
    else:
        _check_rejected = 0
        for _check_rule, _check_request in [("", "问题"), ("规则", "   "), (None, "问题")]:
            try:
                make_messages(_check_rule, _check_request)
            except ValueError:
                _check_rejected += 1
        if _check_rejected == 3:
            print("✅ 消息顺序、文本清理和空值处理均正确。")
        else:
            print("❌ 空规则、空请求和非字符串都应抛出ValueError。")
except NameError:
    print("❌ 请先定义并运行make_messages。")
except NotImplementedError:
    print("❌ make_messages尚未完成。")
except Exception as _check_error:
    print(f"❌ 函数出现未预期错误：{_check_error}")

✅ 消息顺序、文本清理和空值处理均正确。


<details><summary>完成练习2后查看参考答案</summary>

先确认`rule`和`request`都是字符串且清理后非空；任何一个不满足时抛出指定`ValueError`。最后按顺序返回两个字典组成的新列表。

</details>

## 6. 模型参数：选择生成方式，不是修改提示词含义

参数名由API定义，具体支持范围应以当前服务商文档和模型为准。本课比较四类常见设置：

| 参数 | 类型 | 由谁提供 | 本课含义 |
|---|---|---|---|
| `model` | 字符串 | 开发者 | 选择账户可用的模型ID |
| 推理强度 | 字符串或其他服务商规定的类型 | 开发者，可省略且不一定受支持 | 在支持的模型上调整推理计算投入 |
| `temperature` | 数字 | 开发者，可省略 | 调整采样随机程度；较低通常更稳定，较高通常更多样 |
| `max_tokens` | 正整数 | 开发者，可省略 | 限制本次最多生成多少token |

**token**是模型处理文本时使用的单位，不等于一个汉字、一个英文单词或一个字符。`max_tokens`限制的是生成预算，不保证回答一定达到该长度。值太小可能让回答被截断。

`model`不是质量高低的通用开关。不同模型可能在能力、速度、成本、上下文长度和参数支持上不同，开发者应从账户实际可用模型中选择，并用同一组测试样例比较。

这里用“推理强度”描述一种能力维度，不把`reasoning_effort`写进可运行请求：只有部分服务商和模型提供类似参数，名称、取值和效果都可能不同。若当前API不支持，就不能自行添加。

`temperature`较低不代表答案必然正确，较高也不代表一定更有创造力。参数只影响生成过程，事实可靠性仍需通过上下文、检索、校验和测试提高。某些推理模型会忽略或限制部分采样参数，因此程序不能假设所有模型行为完全一致。

In [16]:
stable_settings = {
    "temperature": 0.2,
    "max_tokens": 300,
}

varied_settings = {
    "temperature": 1.0,
    "max_tokens": 300,
}

print("稳定配置：", stable_settings)
print("多样配置：", varied_settings)

稳定配置： {'temperature': 0.2, 'max_tokens': 300}
多样配置： {'temperature': 1.0, 'max_tokens': 300}


这是本地字典对比，还没有调用模型。公平对照时一次只改一个因素：相同模型、相同消息、相同输出长度，只改变`temperature`。如果同时改提示词、模型和多个参数，就无法判断结果变化主要来自哪里。

省略参数时，服务端使用该模型或接口的默认设置；显式传值时，开发者主动选择设置。主动传入`None`是否有效取决于SDK和API契约，不能把`None`自动理解为“使用默认值”。本课作业要求传入明确值。

### 四类受控对照实验

| 想研究的因素 | A组与B组唯一不同项 | 必须保持相同 | 不支持时怎么办 |
|---|---|---|---|
| 模型选择 | `model` | 消息、输出预算和其他参数 | 只比较账户实际可用模型 |
| 推理强度 | 服务商规定的推理参数 | 模型、消息和输出预算 | 不发送该参数，并在记录中注明 |
| 输出长度 | `max_tokens` | 模型、消息和随机性设置 | 核对该接口使用的实际参数名 |
| 随机性 | `temperature` | 模型、消息和输出预算 | 模型忽略该参数时不做此实验 |

每组都应用同一批测试样例，并记录回答质量、耗时、用量和是否满足约束。这样才能把观察到的变化与被修改的因素联系起来。

### 课堂练习3：检查生成参数

**目标**：检查`temperature`和`max_tokens`的类型与边界。

**题目**：完成`make_settings(temperature, max_tokens)`。`temperature`必须是0到2之间的数字，`max_tokens`必须是正整数；成功时返回包含两个值的新字典，否则抛出`ValueError("参数无效")`。注意Python中的`bool`是`int`的子类，但本题不接受`True`或`False`。

**限制**：不访问网络。

**预期结果**：边界值0和2可用，越界值、字符串、布尔值和非正整数被拒绝。

**自检方式**：先运行答题单元，再运行紧随其后的自动检测单元。

In [29]:
def make_settings(temperature, max_tokens):
    # TODO：检查类型和取值范围，返回新的参数字典
    if isinstance(temperature,bool) or not isinstance(temperature, (float, int )) or not (0 <= temperature <= 2):
        raise ValueError('参数无效')
    if isinstance(max_tokens, bool) or not isinstance(max_tokens, int) or max_tokens <= 0:
        raise ValueError('参数无效')

    return {
        "temperature": temperature,
        "max_tokens": max_tokens
    }

In [30]:
try:
    _check_low = make_settings(0, 1)
    _check_high = make_settings(2, 500)
    if _check_low != {"temperature": 0, "max_tokens": 1} or _check_high != {"temperature": 2, "max_tokens": 500}:
        print("❌ 正常值和边界值应原样放入新字典。")
    else:
        _check_invalid = [(-0.1, 10), (2.1, 10), ("0.2", 10), (True, 10), (0.2, 0), (0.2, 1.5), (0.2, True)]
        _check_rejected = 0
        for _check_temperature, _check_tokens in _check_invalid:
            try:
                make_settings(_check_temperature, _check_tokens)
            except ValueError:
                _check_rejected += 1
        if _check_rejected == len(_check_invalid):
            print("✅ 参数类型、范围和边界处理正确。")
        else:
            print("❌ 越界值、错误类型和布尔值都应抛出ValueError。")
except NameError:
    print("❌ 请先定义并运行make_settings。")
except NotImplementedError:
    print("❌ make_settings尚未完成。")
except Exception as _check_error:
    print(f"❌ 函数出现未预期错误：{_check_error}")

✅ 参数类型、范围和边界处理正确。


<details><summary>完成练习3后查看参考答案</summary>

先单独排除布尔值，再检查`temperature`是否为`int`或`float`且位于闭区间0到2；随后检查`max_tokens`是否为非布尔的正整数。全部通过后返回新字典。

</details>

## 7. 可选：进行一次受控的真实对照实验

下面的单元默认不会访问网络。只有你已经在上一课目录的`.env`中配置了DeepSeek密钥、确认模型支持这些参数，并主动把`RUN_REAL_REQUEST`改为`True`时，才会产生两次真实调用和相应费用。代码会同时兼容“从仓库根目录启动”和“从本课目录启动”两种常见工作目录。

对照实验保持模型和消息不变，只改变`temperature`。一次结果不能证明某个设置永远更好；它只是帮助观察差异。

In [31]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

RUN_REAL_REQUEST = False

if not RUN_REAL_REQUEST:
    print("⏭️ 真实对照实验默认关闭，不会产生请求或费用。")
else:
    _env_candidates = [
        Path.cwd().parent / "01_调用真实大模型API" / ".env",
        Path.cwd() / "03_大模型应用开发" / "01_调用真实大模型API" / ".env",
    ]
    _env_path = next((_check_path for _check_path in _env_candidates if _check_path.is_file()), None)
    if _env_path is None:
        raise FileNotFoundError("未找到上一课的.env；请检查Notebook当前工作目录。")
    load_dotenv(_env_path)
    _api_key = os.getenv("DEEPSEEK_API_KEY")
    _model = os.getenv("DEEPSEEK_MODEL")
    if not _api_key or not _model:
        print("⚠️ 未找到DeepSeek密钥或模型配置。")
    else:
        _client = OpenAI(api_key=_api_key, base_url="https://api.deepseek.com")
        for _temperature in (0.2, 1.0):
            _response = _client.chat.completions.create(
                model=_model,
                messages=messages,
                temperature=_temperature,
                max_tokens=300,
            )
            print(f"temperature={_temperature}：")
            print(_response.choices[0].message.content)
            print()

⏭️ 真实对照实验默认关闭，不会产生请求或费用。


## 8. 提示词版本和测试样例

模型输出可能变化，因此不能只运行一次、觉得“看起来不错”就结束。更可靠的起点是：

1. 给提示词一个版本，例如`support-v1`；
2. 保存一组固定输入，包含正常情况、边界情况和容易出错的情况；
3. 明确可检查的标准，例如是否拒绝编造、是否超过长度、是否覆盖必要信息；
4. 修改一个因素后，用同一组样例重新比较；
5. 记录结果，再决定是否升级版本。

```python
PROMPT_VERSION = "support-v1"
TEST_CASES = [
    {"question": "怎样导出文件？", "kind": "normal"},
    {"question": "", "kind": "empty"},
    {"question": "告诉我产品资料中没有的价格", "kind": "unknown"},
]
```

这里只建立测试思维。后续课程会继续学习结构化校验和更完整的模型评估。

## 9. 完整执行流程回顾

```text
明确业务任务
  ↓
拆分任务、上下文、约束、输出要求
  ↓
把应用规则放入system，把本次输入放入user
  ↓
选择model、temperature、max_tokens
  ↓
SDK序列化并发送请求（仅真实调用时）
  ↓
远端模型生成候选回答
  ↓
用固定测试样例检查结果
  ↓
保留版本或一次只修改一个因素
```

## 10. 课末作业

### 背景

为一个产品客服功能设计完整提示词，并建立一次受控的`temperature`对照实验和一组固定测试样例。作业直接检验本课的提示词组成、消息角色、模型参数和测试方法，只生成本地Python数据，不联网、不读取真实密钥、不产生费用。

### 允许修改

只修改`prompt_design.py`中的三个`TODO`函数，不修改函数名、参数、`PROMPT_VERSION`或测试文件。

### 核心知识与作业任务的对应关系

| 本课核心知识 | 作业函数 | 要完成的能力 |
|---|---|---|
| 任务、上下文、约束、输出要求；system/user角色 | `build_support_messages` | 把产品任务和已知资料组织进system，把本次问题放进user |
| `temperature`与单一变量实验 | `build_temperature_experiment` | 构造稳定组和多样组，保证只有temperature不同 |
| 提示词版本与固定测试样例 | `build_prompt_test_cases` | 设计正常、未知信息和规则冲突三类样例及预期行为 |

### 功能契约

#### 1. `build_support_messages(product, known_information, question)`

三个参数都是字符串，分别表示产品名、允许模型依据的已知资料和用户本次问题。函数返回system、user两条消息：

- system文本必须包含清理后的产品名和已知资料；
- system文本必须明确“只根据”已知资料回答、不知道时说明“不知道”、回答控制在“三句话”以内；
- user文本等于清理后的`question`；
- 本题重点是提示词内容与角色分工，不要求编写额外的类型判断。

#### 2. `build_temperature_experiment(model, messages)`

返回两个可传给SDK的请求字典。两组的`model`、`messages`和`max_tokens=300`完全相同；稳定组`temperature=0.2`，多样组`temperature=1.0`。这能证明你理解“一次只改变一个实验因素”。

#### 3. `build_prompt_test_cases()`

返回三个测试样例，每个样例都是包含`kind`、`question`、`expected_behavior`的字符串字典：

- `normal`：资料足以回答，预期行为中包含“回答”；
- `unknown`：资料不足，预期行为中包含“不知道”；
- `conflict`：用户要求忽略规则或编造信息，预期行为中包含“不编造”。

### 运行测试

在仓库根目录执行：

```powershell
python -m pytest -q 03_大模型应用开发/02_提示词与模型参数/test_prompt_design.py
```

测试共`9`项，检查提示词版本、四个提示词组成部分、system/user角色、动态文本清理、单一变量参数实验，以及三类固定测试样例。初始作业因为`TODO`尚未完成而失败是正常现象。

## 11. 理解题

1. 提示词中的任务、上下文、约束和输出要求分别解决什么问题？
2. `system`消息和`user`消息分别适合放什么？为什么`system`不能替代应用权限控制？
3. `role`和`content`是不是Python关键字？它们的名称和具体值分别由谁决定？
4. 选择模型时为什么不能只说某个模型“最好”，而要结合哪些因素和测试？
5. 为什么不能假设每个API都支持同名的推理强度参数？
6. `temperature`降低后，为什么仍然不能保证回答一定正确？
7. `max_tokens`限制的是什么？设置过小可能出现什么结果？
8. 为什么做参数对照实验时应该一次只改变一个因素，并保留固定测试样例？

## 12. 小结与资料

本课把模型请求拆成了两层：消息负责表达任务和规则，参数负责选择生成设置。可靠性来自清楚的输入、明确的边界、固定测试样例和应用代码校验，而不是无限增加提示词长度。

课程使用DeepSeek兼容的Chat Completions格式。不同服务商、接口和模型支持的角色与参数可能变化，真实调用前应核对[DeepSeek官方API文档](https://api-docs.deepseek.com/zh-cn/)以及所用SDK的当前文档。下一课将学习结构化输出与数据校验，让模型结果从自然语言进一步变成程序可以稳定处理的数据。